# Is this the same customer?

*A search finds the few known customers a new record could belong to. A decision model reads the
record beside each of them and says how likely each is to be the same person.*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omnifroodle/couchbase_notebooks/blob/main/notebooks/experiments/02_is_this_the_same_customer.ipynb)
[![Open in Codespaces](https://img.shields.io/badge/Open%20in%20Codespaces-2f363d?logo=github&logoColor=white)](https://codespaces.new/omnifroodle/couchbase_notebooks?quickstart=1)

**Experiment.** Not a finished lesson. The decision model is in limited access behind an alpha
endpoint, and the customers are synthetic, generated for this notebook.
**Claim.** A decision model makes a good judge of whether a new record belongs to a known
customer. It picks the right customer almost every time the search found them, and separates
returning customers from new people far better than the search's own similarity score. It makes
about as many right decisions as the chat model it is compared with, at about a sixteenth of the
cost and under a fortieth of the time.
**Result.** Right customer first for 188 of the 196 returning records the search reached,
against 137 by cosine similarity. On 200 held-out records, 156 right decisions to a chat model's
153, at $0.05 per thousand records and 0.16 s each against $0.76 and 7.5 s. It still scores a son
like his father when the records carry no date of birth.
**Requires.** decision-model · local-embeddings · llm
**Read** ~12 min · **Run** ~5 min · **Cost** ~$0.25, nearly all of it the chat-model comparison

Sooner or later, every customer system has to answer one question: *is this someone we already
know?* A sign-up, an order or a support ticket arrives with a name and whatever else the form
asked for. Either it belongs to an existing customer or it is someone new, and both mistakes are
expensive. Miss the match and one customer has two accounts, two loyalty balances and two
marketing profiles. Make a false match and you have merged two people, a son into his father's
account, and shown one of them the other's orders.

The field calls this **entity resolution**, and it splits into two steps worth keeping apart:

1. **Find candidates.** Search the known customers for the handful who could plausibly be this
   person. This step runs against everyone, so it has to be cheap, and anyone it misses is lost.
2. **Judge each candidate.** Read the new record beside each candidate and decide. This step only
   ever sees a handful, so it can afford to be careful.

Hand-written rules and string-similarity scores have done the judging for decades. This notebook
tries a **decision model** as the judge: [TypeSafe's Jev](https://typesafe.ai/blog/introducing-system-one-models-and-jev),
introduced in [a first look at decision models](01_first_look_at_decision_models.ipynb), which
answers a yes/no question with a probability in a fraction of a second. It is compared with the
similarity score the search already produced, and with a general chat model given the same
records.

**Setup.** An [OpenRouter API key](https://openrouter.ai/settings/keys) with access to
`typesafe/jev` in `OPENROUTER_API_KEY`, and whichever chat model the other notebooks use. No
Couchbase cluster is needed: the search in section 2 runs in memory, so that the judge can be
looked at on its own.

In [ ]:
# Setup: load the helpers, install anything missing, check what this notebook needs.
import os
import pathlib
import subprocess
import sys

# Cloned on Colab, where there is no local checkout. Override to test a fork.
REPO_URL = os.environ.get("CBNB_REPO_URL", "https://github.com/omnifroodle/couchbase_notebooks")

try:
    import cbnb
except ModuleNotFoundError:
    here = pathlib.Path.cwd()
    root = next((p for p in [here, *here.parents] if (p / "cbnb" / "__init__.py").exists()), None)
    if root is None:
        # Colab: clone the repo so the committed datasets come with it. Once: a
        # restarted session keeps the files but not sys.path.
        root = pathlib.Path("cbnb-repo").resolve()
        if not root.exists():
            subprocess.check_call(["git", "clone", "--depth", "1", "--quiet", REPO_URL, str(root)])
    sys.path.insert(0, str(root))
    import cbnb

settings = cbnb.bootstrap(requires=["decision-model", "local-embeddings", "llm"])

## 1. Known customers, and the records arriving

Real customer data cannot be published, and public entity-resolution benchmarks are mostly about
products or bibliographies. So these customers are generated, by
[`cbnb/profiles.py`](../../cbnb/profiles.py): 1,000 known customers as a CRM would hold them, and
400 incoming records whose true owner is known.

Every name is real and every person is invented. Emails use the reserved `.example` domain and
phone numbers the 555-01xx range kept for fiction, so no record reaches a real inbox or phone.

In [2]:
# Load the known customers and the incoming records, and count the incoming ones by kind.
import pandas as pd

from cbnb.profiles import FIELDS, RELATIONS, VARIATIONS, load_profiles

customers, incoming = load_profiles()
on_file = customers.set_index("customer_id")
print(f"{len(customers):,} known customers, {len(incoming)} incoming records")
incoming.truth_kind.value_counts().rename("incoming records").to_frame()

1,000 known customers, 400 incoming records


,incoming records
truth_kind,
returning,220
relative,80
stranger,60
namesake,40


Each incoming record is one of four kinds. **Returning** records belong to a known customer and
should be linked. The other three are new people and must not be linked:

- **relative**: someone in a known customer's household, sharing a surname and often an address
  and phone;
- **namesake**: someone with a known customer's exact name and nothing else in common;
- **stranger**: someone unrelated to anyone.

Each `truth_` column holds the answer. Nothing a matcher sees is taken from those columns.

Here is a returning customer. The record arrived through a different channel from the one the
customer signed up through, so it carries different fields, and a couple of things have changed.

In [3]:
# Show a returning customer's new record beside the customer on file.
def side_by_side(record, customer_id):
    return pd.DataFrame({"new record": record[FIELDS],
                         f"on file as {customer_id}": on_file.loc[customer_id, FIELDS]}).fillna("")

example_id = incoming.index[incoming.record_id == "N0177"][0]
print("what changed:", ", ".join(incoming.truth_variations[example_id]))
side_by_side(incoming.loc[example_id], incoming.truth_customer_id[example_id])

what changed: nickname, swapped


,new record,on file as C0527
first_name,Baker,Dorothy
last_name,Dot,Baker
dob,,1946-04-25
email,,dorothy.b@mail.example
phone,(505) 555-0155,(505) 555-0155
street,,2408 Prospect Pl
city,,Albuquerque
state,,NM
zip,87104,87104


Here is a new person who is not a customer but lives with one: a son given his father's name.
Their emails differ, but so do many returning customers'. When both records carry a date of
birth, it is the one field that reliably separates them.

In [4]:
# Show a son who shares his father's home beside the father, who is already a customer.
def shares_home(record):
    father = on_file.loc[record.truth_related_to]
    return any(pd.notna(record[f]) and record[f] == father[f] for f in ("street", "zip", "phone"))


sons = incoming[(incoming.truth_relation == "junior") & incoming.dob.notna()]
son_id = sons.index[sons.apply(shares_home, axis=1)][0]
side_by_side(incoming.loc[son_id], incoming.truth_related_to[son_id])

,new record,on file as C0933
first_name,James,James
last_name,Ramirez,Ramirez
dob,1972-12-06,1948-06-18
email,james.ramirez@letters.example,james.ramirez660@inbox.example
phone,(617) 555-0189,(617) 555-0189
street,,7825 Pine Blvd
city,Boston,Boston
state,MA,MA
zip,,02134


Returning customers change in the ways below, one to three at a time. The rates are the
generator's choices, not facts about any real customer base, and they drive every aggregate in
this notebook. That is why the results below are broken down by kind.

In [5]:
# The ways a returning customer's record can differ from the one on file, and how often each occurs.
from collections import Counter

counts = Counter(change for changes in incoming.truth_variations for change in changes)
pd.DataFrame({"what changed": VARIATIONS, "records": pd.Series(counts)}).fillna(0).astype({"records": int})

,what changed,records
dob_swapped,Day and month of birth transposed (only when b...,7
initial,First name reduced to an initial (Robert -> R.).,38
moved,"A new street address, often in another city; h...",34
new_email,A different email address.,31
new_phone,A different phone number.,33
new_surname,"Surname changed, or double-barrelled, as on ma...",50
nickname,First name replaced by a common nickname (Robe...,45
swapped,First and last name entered in each other's fi...,53
typo,"One slip in the first or last name: a dropped,...",49


## 2. Finding candidates

Candidates come from two searches, and are merged:

- **Nearest neighbours.** Each record is written as its values joined by ` | `, embedded with a
  small local model, and compared with every known customer. The five most similar are kept. This
  finds customers whose name is misspelt, abbreviated or swapped.
- **Exact keys.** Any known customer with exactly the same email or phone is added. This finds
  customers whose name has changed but whose contact details have not.

In production both are queries against the profile collection: a vector search, and a lookup on
an indexed field.

In [6]:
# Write a record as its values joined by " | ", leaving out the fields it doesn't have.
def present(record):
    # The fields that have a value, in display order.
    return {field: record[field] for field in FIELDS if pd.notna(record[field])}


def as_values(record):
    return " | ".join(present(record).values())

print(as_values(incoming.loc[example_id]))
print(as_values(on_file.loc[incoming.truth_customer_id[example_id]]))

Baker | Dot | (505) 555-0155 | 87104
Dorothy | Baker | 1946-04-25 | dorothy.b@mail.example | (505) 555-0155 | 2408 Prospect Pl | Albuquerque | NM | 87104


In [7]:
# Embed both sets on this machine, and keep each incoming record's five nearest known customers.
import numpy as np

from cbnb.embeddings import Embedder

embedder = Embedder(backend="local")
known_vectors = embedder.encode([as_values(row) for _, row in customers.iterrows()])
incoming_vectors = embedder.encode([as_values(row) for _, row in incoming.iterrows()])
similarity = incoming_vectors @ known_vectors.T   # cosine: the vectors are unit length
nearest = np.argsort(-similarity, axis=1)[:, :5]

In [8]:
# Add every known customer sharing an exact email or phone, then check what the judge will be able to reach.
customer_ids = customers.customer_id.to_numpy()
by_key = {}
for position, row in enumerate(customers.itertuples()):
    for key in (row.email, row.phone):
        if pd.notna(key):
            by_key.setdefault(key, []).append(position)

position = {cid: p for p, cid in enumerate(customer_ids)}
candidates, from_vectors = [], []
for i, row in enumerate(incoming.itertuples()):
    exact = [p for key in (row.email, row.phone) if pd.notna(key) for p in by_key.get(key, [])]
    candidates.append(list(customer_ids[list(dict.fromkeys([*nearest[i], *exact]))]))
    from_vectors.append(set(customer_ids[nearest[i]]))
incoming["candidates"] = candidates

returning = incoming.truth_kind == "returning"
in_reach = incoming.apply(lambda r: r.truth_customer_id in r.candidates, axis=1)
by_vectors = pd.Series([t in c for t, c in zip(incoming.truth_customer_id, from_vectors)], incoming.index)
print(f"{incoming.candidates.map(len).mean():.1f} candidates per record on average, "
      f"{incoming.candidates.map(len).max()} at most")
print(f"true customer among the candidates, for {returning.sum()} returning records: "
      f"{by_vectors[returning].mean():.0%} from nearest neighbours alone, "
      f"{in_reach[returning].mean():.0%} with exact keys added")
print("out of reach, by the source of the record:",
      incoming.source[returning & ~in_reach].value_counts().to_dict())

5.1 candidates per record on average, 7 at most
true customer among the candidates, for 220 returning records: 77% from nearest neighbours alone, 89% with exact keys added
out of reach, by the source of the record: {'support': 14, 'signup': 5, 'store': 4, 'order': 1}


That second number is a ceiling. A returning customer who is not among their record's candidates
cannot be linked by any judge, however good. More than half of those misses are support tickets,
which carry a name and an email and nothing else. Once the email has changed too, only a name is
left to search on. Reaching them takes a better candidate search, which is a different notebook's
subject. The rest of this notebook is about the records that *are* in reach.

## 3. Asking the judge

One call per incoming record. The state is the new record followed by its numbered candidates,
and there is one yes/no question per candidate. The questions are answered independently, so two
candidates can both score high. That happens when a father and a son are both customers and the
new record could be either of them. A single-choice question would have to split its
probability between them.

Asking several questions in one call costs no more time than asking one (see
[the first look](01_first_look_at_decision_models.ipynb)). The
question ids are the customer ids, for this code only. The model sees the instructions and the
state.

In [9]:
# Build the judge's input: the new record and its numbered candidates, and one question per candidate.
from cbnb.decisions import choice, decide, noul

SAME = ("The same individual. Names may be nicknames, initials, misspelt or entered in the wrong "
        "field, and a surname, address, email or phone may have changed.")
DIFFERENT = ("A different individual, even one who shares a name, address or phone with the new "
             "record, such as a relative in the same household.")


def judge_state(record, write=as_values):
    listing = "\n".join(f"Customer {n}: {write(on_file.loc[cid])}"
                        for n, cid in enumerate(record.candidates, 1))
    return f"New record: {write(record)}\n\n{listing}"


def judge_questions(record):
    return {cid: noul(f"Is customer {n} the same person as the new record?", true=SAME, false=DIFFERENT)
            for n, cid in enumerate(record.candidates, 1)}


print(judge_state(incoming.loc[example_id]))

New record: Baker | Dot | (505) 555-0155 | 87104

Customer 1: William | Baker | 1955-10-14 | williambaker880@mail.example | (413) 555-0138 | 7247 Spring Pl | Springfield | MA | 01169
Customer 2: William | Baker | 1996-08-09 | william_baker96@inbox.example | (971) 555-0189 | 8820 Sunset Pl | Portland | OR | 97251
Customer 3: Luis | Baker | baker.luis46@mail.example | (412) 555-0119 | 4009 Madison Ave | Pittsburgh | PA | 15222
Customer 4: James | Baker | 1987-11-10 | james.baker@post.example | (305) 555-0132 | 1888 Lincoln Pl | Miami | FL | 33177
Customer 5: Melissa | Baker | 1959-12-20 | melissa_baker@post.example | (413) 555-0138 | 7247 Spring Pl | Springfield | MA | 01169
Customer 6: Dorothy | Baker | 1946-04-25 | dorothy.b@mail.example | (505) 555-0155 | 2408 Prospect Pl | Albuquerque | NM | 87104


In [10]:
# Ask about the example, and put each candidate's probability beside its cosine similarity.
example = incoming.loc[example_id]
answer = decide(judge_state(example), judge_questions(example))
row = incoming.index.get_loc(example_id)
pd.DataFrame({
    "candidate": example.candidates,
    "cosine": [similarity[row, position[c]] for c in example.candidates],
    "same person? (Jev)": [answer.value(c) for c in example.candidates],
    "is the customer": [c == example.truth_customer_id for c in example.candidates],
}).style.format({"cosine": "{:.2f}", "same person? (Jev)": "{:.2f}"}).hide(axis="index")

candidate,cosine,same person? (Jev),is the customer
C0284,0.65,0.10,False
C0079,0.64,0.09,False
C0169,0.63,0.07,False
C0403,0.63,0.05,False
C0413,0.63,0.10,False
C0527,0.53,0.86,True


Now every incoming record, eight calls in flight at a time. Each record is sent twice. The
second time, it is written as sentences instead of values ("Their name is Bob Allen. They live
at…"), to see whether the judge cares how a record is written.

In [11]:
# Write a record as short sentences instead, leaving out the fields it doesn't have.
from datetime import date


def as_sentences(record):
    r = dict.fromkeys(FIELDS) | present(record)
    sentences = []
    if name := " ".join(part for part in (r["first_name"], r["last_name"]) if part):
        sentences.append(f"Their name is {name}.")
    if r["dob"]:
        born = date.fromisoformat(r["dob"])
        sentences.append(f"They were born on {born.day} {born:%B %Y}.")
    region = " ".join(part for part in (r["state"], r["zip"]) if part)
    if place := ", ".join(part for part in (r["street"], r["city"], region) if part):
        sentences.append(f"They live {'at' if r['street'] else 'in'} {place}.")
    if r["email"]:
        sentences.append(f"Their email address is {r['email']}.")
    if r["phone"]:
        sentences.append(f"Their phone number is {r['phone']}.")
    return " ".join(sentences)


print(as_sentences(incoming.loc[example_id]))

Their name is Baker Dot. They live in 87104. Their phone number is (505) 555-0155.


In [12]:
# Ask about every incoming record, written both ways: one call per record, eight at a time.
import statistics
import time
from concurrent.futures import ThreadPoolExecutor

import httpx


def ask_everyone(make_questions, write=as_values):
    records = [record for _, record in incoming.iterrows()]
    with httpx.Client() as client, ThreadPoolExecutor(8) as pool:
        decide("Warm-up.", {"x": noul("Is this a warm-up?")}, client=client)  # opens the connection
        return list(pool.map(lambda r: decide(judge_state(r, write), make_questions(r), client=client),
                             records))


judged = {}
for name, write in [("values", as_values), ("sentences", as_sentences)]:
    started = time.perf_counter()
    judged[name] = ask_everyone(judge_questions, write)
    calls = judged[name]
    print(f"written as {name}: {len(calls)} calls in {time.perf_counter() - started:.0f} s, "
          f"median {statistics.median(c.seconds for c in calls) * 1000:.0f} ms per call, "
          f"${sum(c.cost for c in calls):.4f} in all, {sum(c.attempts > 1 for c in calls)} retried")

written as values: 400 calls in 9 s, median 159 ms per call, $0.0184 in all, 0 retried


written as sentences: 400 calls in 10 s, median 171 ms per call, $0.0197 in all, 0 retried


## 4. Did it pick the right customer?

Every method is reduced to the same two things per record: its **top pick** among the
candidates, and a **score** for how strongly it backs that pick. For cosine similarity the score
is the similarity. For Jev it is the probability that the top pick is the same person.

First, the top pick alone, on the returning records whose customer is in reach.

In [13]:
# Each method's top pick and score, for every incoming record.
picks = pd.DataFrame(index=incoming.index)


def add_method(name, scores_per_record):
    best = [max(scores, key=scores.get) for scores in scores_per_record]
    picks[name] = best
    picks[f"{name} score"] = [scores[b] for scores, b in zip(scores_per_record, best)]


add_method("cosine", [{c: float(similarity[i, position[c]]) for c in record.candidates}
                      for i, (_, record) in enumerate(incoming.iterrows())])
for name, calls in judged.items():
    add_method(f"jev ({name})", [{c: call.value(c) for c in record.candidates}
                                 for call, (_, record) in zip(calls, incoming.iterrows())])

reachable = returning & in_reach
methods = ["cosine", "jev (values)", "jev (sentences)"]
pd.DataFrame({
    "top pick is the customer": [
        f"{(picks.loc[reachable, m] == incoming.truth_customer_id[reachable]).sum()} of {reachable.sum()}"
        for m in methods]}, index=methods)

,top pick is the customer
cosine,137 of 196
jev (values),188 of 196
jev (sentences),189 of 196


The judge fixes most of the search's ranking mistakes. Cosine similarity put the right
customer first for 137 of the 196 returning records in reach, and Jev picked the right one for
188. The example above shows the pattern: the search ranked five other Bakers above Dorothy, and
Jev gave Dorothy 0.86 and none of the others more than 0.10.

Writing the records as sentences made little difference to the judge: 188 as values, 189 as
sentences. The rest of
this notebook uses the values.

## 5. Should it link at all?

Picking is half the decision. A new person has candidates too, and one of them is always on top,
so the score has to do the other half: high for a returning customer, low for someone new. The
table shows each kind's median score with the 10th and 90th percentiles either side. The last
row is a single number for how well the score separates the two groups: the chance that a
returning customer whose top pick is right outscores a new person.

In [14]:
# Score ranges by kind, and how well each method's score separates returning customers from new people.
labels = {"junior": "relative: child, same name", "senior": "relative: parent, same name",
          "spouse": "relative: spouse", "twin": "relative: twin"}
kind = incoming.truth_kind.where(incoming.truth_kind != "relative", incoming.truth_relation.map(labels))
new_person = incoming.truth_customer_id.isna()


def spread(scores):
    low, mid, high = np.quantile(scores, [0.1, 0.5, 0.9])
    return f"{low:.2f} · {mid:.2f} · {high:.2f}"


def separation(method):
    right = picks.loc[returning & (picks[method] == incoming.truth_customer_id), f"{method} score"].to_numpy()
    new = picks.loc[new_person, f"{method} score"].to_numpy()
    return f"{((right[:, None] > new) + 0.5 * (right[:, None] == new)).mean():.2f}"


order = ["returning", *labels.values(), "namesake", "stranger"]
table = pd.DataFrame({m: picks.groupby(kind)[f"{m} score"].apply(spread) for m in methods}).loc[order]
table.insert(0, "records", kind.value_counts())
table.loc["separation"] = ["", *[separation(m) for m in methods]]
table

,records,cosine,jev (values),jev (sentences)
truth_kind,,,,
returning,220,0.61 · 0.75 · 0.88,0.43 · 0.87 · 0.95,0.37 · 0.84 · 0.95
"relative: child, same name",27,0.67 · 0.78 · 0.89,0.38 · 0.73 · 0.92,0.20 · 0.69 · 0.92
"relative: parent, same name",22,0.71 · 0.81 · 0.94,0.29 · 0.82 · 0.92,0.12 · 0.73 · 0.88
relative: spouse,19,0.64 · 0.72 · 0.82,0.19 · 0.31 · 0.61,0.09 · 0.19 · 0.64
relative: twin,12,0.64 · 0.69 · 0.79,0.20 · 0.32 · 0.54,0.12 · 0.27 · 0.41
namesake,40,0.66 · 0.75 · 0.81,0.30 · 0.55 · 0.70,0.14 · 0.44 · 0.73
stranger,60,0.63 · 0.72 · 0.80,0.17 · 0.28 · 0.56,0.07 · 0.12 · 0.43
separation,,0.63,0.90,0.90


The cosine column barely changes from row to row. A stranger's best candidate scores 0.72 at
the median, a returning customer's 0.75, and the ranges overlap almost entirely: the score that
found a candidate is a poor rule for whether to link it. Jev's scores spread out, from 0.28 for
strangers and about a third for spouses and twins to 0.87 for returning customers. Two groups
stay high. Namesakes sit in the middle at 0.55, because a sparse record often carries little
more than the name. Same-name parents and children score almost like returning customers, at
0.73 and 0.82.

Those parents and children are the hardest case in the table, and the records show why. The one
field that tells them apart is the date of birth, and most channels don't ask for it.

In [15]:
# Jev's score for same-name parents and children, split by whether both records carry a date of birth.
same_name = incoming.truth_relation.isin(["junior", "senior"])
picked_dob = on_file.loc[picks["jev (values)"], "dob"].notna().to_numpy()
both = pd.Series(incoming.dob.notna().to_numpy() & picked_dob, incoming.index)
evidence = both.map({True: "both have a date of birth", False: "one or neither has one"})
(picks.loc[same_name, "jev (values) score"]
 .groupby(evidence[same_name]).agg(records="size", median="median", highest="max").round(2))

,records,median,highest
both have a date of birth,8,0.39,0.89
one or neither has one,41,0.80,0.95


In [16]:
# The same-name records where both sides have a date of birth: how far apart the dates are, and Jev's score.
dated = incoming[same_name & both]
pd.DataFrame({
    "new record born": dated.dob,
    "customer picked, born": on_file.loc[picks.loc[dated.index, "jev (values)"], "dob"].to_numpy(),
    "years apart": [abs(int(a[:4]) - int(b[:4])) for a, b in
                    zip(dated.dob, on_file.loc[picks.loc[dated.index, "jev (values)"], "dob"])],
    "Jev: same person?": picks.loc[dated.index, "jev (values) score"],
}).sort_values("Jev: same person?", ascending=False).style.format(precision=2).hide(axis="index")

new record born,"customer picked, born",years apart,Jev: same person?
1971-08-19,2005-09-26,34,0.89
1950-08-04,1976-10-10,26,0.69
1972-12-06,1948-06-18,24,0.68
2000-07-21,1986-05-09,14,0.40
1957-12-23,1985-05-06,28,0.38
1988-01-10,1965-10-29,23,0.35
1956-02-21,1956-07-11,0,0.28
1945-05-27,1940-04-14,5,0.27


Of the 49 same-name parents and children, 41 are missing a date of birth on one side or both,
and Jev scores them almost like the customer (median 0.80). Without a date, such a record looks
exactly like a returning customer: the same name, and either the same home or a different one.
Returning customers move, and change their email and phone, too. Nothing these records carry can
tell the two apart.

Where both records have a date of birth, the median falls to 0.39, but not to zero. In the
second table, three pairs born 24 to 34 years apart still score between 0.68 and 0.89. A decision
model reads a nickname or a swapped field well. Weighing one date against another is arithmetic,
and it does that less reliably.

Neither problem needs a better model. Collect the date of birth where households matter, let
code compare the dates when both are there, and send a same-surname, same-address match with no
date to a person.

## 6. Compared with a chat model, and with a single choice

Two more ways of asking the same thing, each as one call per record:

- **Jev, pick one.** A `choice` question whose options are the numbered candidates and `none`.
  One answer per record, with a probability spread over the options.
- **A chat model, pick one.** The same state and the same instructions, answered in JSON by
  whichever chat model this environment is configured for.

Both give a decision outright. The two score-based methods need a threshold: link the top pick
if its score clears it. To keep that honest, the incoming records are split in half with the mix
of kinds preserved. Each threshold is set on one half, and every method is scored on the other.

In [17]:
# Split the incoming records in half, keeping the mix of kinds: tune on one half, report on the other.
tuning = incoming.groupby("truth_kind", group_keys=False).sample(frac=0.5, random_state=0).index
held_out = incoming.index.difference(tuning)


def outcome(linked, truth):
    if pd.isna(linked):
        return "right: new person" if pd.isna(truth) else "missed a customer"
    if linked == truth:
        return "right: linked"
    return "linked a new person" if pd.isna(truth) else "linked the wrong customer"


def outcomes(linked, index):
    return pd.Series([outcome(linked[i], incoming.truth_customer_id[i]) for i in index], index=index)


def linked_at(method, threshold):
    return picks[method].where(picks[f"{method} score"] >= threshold, None)


def best_threshold(method):
    scores = sorted(set(picks.loc[tuning, f"{method} score"]))
    return max(scores, key=lambda t: outcomes(linked_at(method, t), tuning).str.startswith("right").sum())


thresholds = {m: best_threshold(m) for m in ["cosine", "jev (values)"]}
thresholds

{'cosine': 0.8111587166786194, 'jev (values)': 0.7}

In [18]:
# Ask Jev to pick one candidate, or none, for each held-out record.
def pick_question(record):
    options = {str(n): None for n in range(1, len(record.candidates) + 1)}
    options["none"] = "None of them. " + DIFFERENT
    return {"pick": choice(f"Which customer is the same person as the new record? {SAME}", options)}


held_records = [incoming.loc[i] for i in held_out]
with httpx.Client() as client, ThreadPoolExecutor(8) as pool:
    picked = list(pool.map(lambda r: decide(judge_state(r), pick_question(r), client=client), held_records))
jev_pick = pd.Series([None if c.value("pick") == "none" else r.candidates[int(c.value("pick")) - 1]
                      for c, r in zip(picked, held_records)], index=held_out, dtype=object)
print(f"{len(picked)} calls, median {statistics.median(c.seconds for c in picked) * 1000:.0f} ms, "
      f"${sum(c.cost for c in picked):.4f} in all")

200 calls, median 162 ms, $0.0062 in all


In [19]:
# Ask the chat model the same thing for each held-out record, and time every call.
import re

from pydantic import BaseModel

from cbnb.llm import LLM


class Pick(BaseModel):
    customer: str


INSTRUCTIONS = ("You match customer records. You are given a new record and a numbered list of "
                "customers on file. Answer with the number of the customer who is the same person "
                f"as the new record, or \"none\" if none of them is. Same person: {SAME} "
                f"Different person: {DIFFERENT}")

llm = LLM(cache=False)   # every call is real, so the timings are too


def ask_chat(record):
    started = time.perf_counter()
    reply = llm.structured(judge_state(record), Pick, INSTRUCTIONS).customer.strip().lower()
    number = re.search(r"\d+", reply)
    n = int(number.group()) if number else 0
    return (record.candidates[n - 1] if 1 <= n <= len(record.candidates) else None,
            reply, time.perf_counter() - started)


chatted = llm.map(held_records, ask_chat, progress=False)
chat_pick = pd.Series([linked for linked, _, _ in chatted], index=held_out, dtype=object)
unreadable = sum("none" not in reply and not re.search(r"\d", reply) for _, reply, _ in chatted)
print(f"{llm.model}: {len(chatted)} calls, {llm.usage.prompt_tokens / len(chatted):.0f} prompt and "
      f"{llm.usage.completion_tokens / len(chatted):.0f} completion tokens per record, median "
      f"{statistics.median(s for _, _, s in chatted):.1f} s per call; {unreadable} answers were "
      "neither a number nor 'none'")

z-ai/glm-5.3: 200 calls, 462 prompt and 26 completion tokens per record, median 7.5 s per call; 0 answers were neither a number nor 'none'


Cost and speed per record. Jev's cost is what OpenRouter reported for these calls. The chat
model reports tokens, not dollars, so its cost comes from the price per million tokens set in the
cell. Change that price for your model: the listed one is for the model that produced this run.

In [20]:
# Right and wrong decisions on the held-out half, with the time and cost of each method per record.
CHAT_PRICE = {"model": "z-ai/glm-5.3", "input": 1.40, "output": 4.40}  # $ per million tokens, OpenRouter's list price, 2026-09-29

held_calls = [judged["values"][incoming.index.get_loc(i)] for i in held_out]
chat_cost = (llm.usage.prompt_tokens * CHAT_PRICE["input"] + llm.usage.completion_tokens * CHAT_PRICE["output"]) / 1e6
rows = {
    f"cosine ≥ {thresholds['cosine']:.2f}": (linked_at("cosine", thresholds["cosine"]), None, 0.0),
    f"Jev, yes/no per candidate ≥ {thresholds['jev (values)']:.2f}": (
        linked_at("jev (values)", thresholds["jev (values)"]),
        statistics.median(c.seconds for c in held_calls), sum(c.cost for c in held_calls)),
    "Jev, pick one": (jev_pick, statistics.median(c.seconds for c in picked), sum(c.cost for c in picked)),
    f"chat model ({llm.model}), pick one": (
        chat_pick, statistics.median(s for _, _, s in chatted),
        chat_cost if llm.model == CHAT_PRICE["model"] else float("nan")),
}
columns = ["right: linked", "right: new person", "missed a customer", "linked the wrong customer",
           "linked a new person"]
compare = pd.DataFrame({name: outcomes(linked, held_out).value_counts() for name, (linked, _, _) in rows.items()}).T
compare = compare.reindex(columns=columns).fillna(0).astype(int)
compare["seconds per record"] = [s for _, s, _ in rows.values()]
compare["$ per 1,000 records"] = [c / len(held_out) * 1000 for _, _, c in rows.values()]
compare.style.format({
    "seconds per record": lambda s: "local" if s is None or pd.isna(s) else f"{s:.2f}",
    "$ per 1,000 records": lambda c: "price not set" if pd.isna(c) else ("local" if c == 0 else f"{c:.3f}"),
})

,right: linked,right: new person,missed a customer,linked the wrong customer,linked a new person,seconds per record,"$ per 1,000 records"
cosine ≥ 0.81,31,76,77,2,14,local,local
"Jev, yes/no per candidate ≥ 0.70",81,75,27,2,15,0.16,0.046
"Jev, pick one",86,33,13,11,57,0.16,0.031
"chat model (z-ai/glm-5.3), pick one",88,65,19,3,25,7.51,0.762


With its threshold, Jev's yes/no questions make 156 right decisions out of 200, and the chat
model makes 153. They go wrong in different directions. Jev misses more returning customers (27
against 19) and links fewer new people (15 against 25), and moving the threshold trades one kind
of error for the other. On cost and speed they are far apart: Jev answered in 0.16 s at $0.05
per thousand records, the chat model in 7.5 s at $0.76.

Asking Jev to pick one candidate is cheaper still, and clearly worse: it linked 57 of the 90 new
people. Separate yes/no questions can all come back no, which is the answer a new person needs.

## 7. Three answers instead of two: link, ask a person, or new customer

A probability supports a third answer. Link when the score is high, create a new customer when it
is low, and send the middle band to a person. Record linkage has made decisions this way since
Fellegi and Sunter formalised it in 1969. The model's contribution is to make the band
narrower.

The two thresholds are set on the tuning half so that no more than 2% of the automatic
decisions on either side are wrong. On the linking side, wrong means linking the wrong person.
On the new-customer side, wrong means turning away a customer who was among the candidates.
Returning records whose customer never reached the candidates are counted separately, because
no threshold can help them.

In [21]:
# Pick both thresholds on the tuning half, then count what each band would hold on the held-out half.
BUDGET = 0.02


def bands(method):
    score, pick = picks[f"{method} score"], picks[method]
    wrong_link = pick != incoming.truth_customer_id
    missed = in_reach & returning                 # a customer the judge could have linked
    grid = sorted(set(score[tuning]))
    high = min((t for t in grid if wrong_link[tuning][score[tuning] >= t].mean() <= BUDGET),
               default=float("inf"))
    low = max([t for t in grid if t < high and missed[tuning][score[tuning] <= t].mean() <= BUDGET],
              default=-1.0)
    held = score[held_out]
    link, new = held >= high, held <= low
    return {
        "thresholds": f"link ≥ {high:.2f}, new ≤ {low:.2f}" if low >= 0 else f"link ≥ {high:.2f}, new: never",
        "linked": f"{link.sum()} ({wrong_link[held_out][link].sum()} wrong)",
        "sent to a person": int((~link & ~new).sum()),
        "new customer": f"{new.sum()} ({missed[held_out][new].sum()} were customers in reach)",
        "customers out of reach": int((returning & ~in_reach)[held_out].sum()),
    }


pd.DataFrame({m: bands(m) for m in ["cosine", "jev (values)"]}).T

,thresholds,linked,sent to a person,new customer,customers out of reach
cosine,"link ≥ 0.97, new ≤ 0.48",1 (0 wrong),199,0 (0 were customers in reach),12
jev (values),"link ≥ 0.95, new ≤ 0.33",19 (1 wrong),135,46 (3 were customers in reach),12


Cosine similarity cannot decide anything on its own. To stay within the budget its link
threshold rises to 0.97, and all but one record ends up with a person. Jev decides 65 of the
200 automatically, with 4 mistakes, and sends 135 to a person.

Jev's link threshold is pushed up to 0.95 by the same-name relatives without a birth date, who
score as high as 0.95 themselves. This data over-represents them on purpose, so the automatic
share here says little about a real customer base. The method carries over: set the thresholds
on your own labelled records, and measure how wide the middle band is.

## Where to take this

- **Candidates on Couchbase.** Put the profiles in a collection, find candidates with a vector
  search plus an index lookup on email and phone, and measure what this notebook took as given:
  which text to embed, and whether a fuzzy text search on names reaches the records that
  embeddings miss.
- **Resolve on write.** A candidate search plus one decision takes a fraction of a second, which
  is fast enough to run inside a sign-up request, so a customer's second account is never
  created.
- **Let code compare the dates.** Two birth dates decades apart settle the question, and a rule
  can say so before the model is asked. The model then only judges what rules handle badly:
  nicknames, typos, fields in the wrong place.
- **Ask about the household too.** A second yes/no question in the same call, "does the new
  record share a household with this customer?", costs no extra time. Its answer can route
  a record to a person instead of letting a son be linked to his father.
- **From pairs to people.** Decisions are pairwise, and customers are not. If A matches B and B
  matches C, is A the same person as C? That becomes a data-model question about one document per
  person, with the records it absorbed.

## What produced these results

Every number above came from the models below. A different model, or the same name served
differently, will give different numbers.


In [22]:
# Name the models that produced the results above.
cbnb.run_card()

,model,how it ran
Chat model,z-ai/glm-5.3,via NanoGPT; 200 calls
Embeddings,sentence-transformers/all-MiniLM-L6-v2,"on this machine; 1,400 texts, 1,400 from cache"
Decision model,typesafe/jev-1.13-20260917,"via OpenRouter; 1,003 calls"
